<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day3/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Prétraitement et optimisation des modèles basés sur les transformateurs

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
import tensorflow as tf
from transformers import BertTokenizer, XLMRobertaTokenizer

# Fixer les graines aléatoires pour la reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# =====================================================================
# 1 & 4. CHARGEMENT ET EXPLORATION DE L'ENSEMBLE DE DONNÉES (SIMULATION)
# =====================================================================
print("--- Étape 1 & 4 : Chargement et Exploration du jeu de données ---")

# Simulation d'un fichier CSV avec suffisamment d'exemples par classe
mock_data = pd.DataFrame({
    'text': [
        "L'équipe a remporté la coupe du monde après un match légendaire.",
        "Le cours de cette action en bourse a chuté de 5% ce matin.",
        "Le nouveau modèle de smartphone intègre une puce d'intelligence artificielle.",
        "Le joueur a marqué un but décisif à la dernière minute du jeu.",
        "Les investisseurs craignent une hausse des taux d'intérêt.",
        "L'algorithme de deep learning optimise le traitement des images médicales.",
        "Le président du club de football a annoncé le transfert de l'attaquant.",
        "La start-up technologique a levé 10 millions d'euros pour son logiciel.",
        "La mise à jour du système d'exploitation corrige une faille de sécurité.",
        "Le marathon de Paris a réuni plus de cinquante mille coureurs cette année.",
        "L'attaquant vedette a signé un contrat de trois ans avec son nouveau club.",
        "L'indice boursier a atteint un sommet historique en fin de séance.",
        "Le cloud computing révolutionne le stockage des données en entreprise.",
        "Le match de tennis a été interrompu par la pluie au troisième set.",
        "L'inflation impacte directement le pouvoir d'achat des ménages."
    ],
    'label': [0, 1, 2, 0, 1, 2, 0, 2, 2, 0, 0, 1, 2, 0, 1]  # 0: Sport, 1: Finance, 2: Technologie
})

# Écriture locale pour mimer le chargement d'un fichier CSV externe
os.makedirs('./data', exist_ok=True)
mock_data.to_csv('./data/train_dataset.csv', index=False)

# Chargement officiel requis par la consigne
df = pd.read_csv('./data/train_dataset.csv')

print(f"Forme du jeu de données (Shape) : {df.shape}")
print("\nAperçu des premières lignes :")
display(df.head())

# =====================================================================
# 2 & 3. TOKENISATION ET PRÉPARATION DES DONNÉES D'ENTRÉE (BERT & XLM-R)
# =====================================================================
print("\n--- Étape 2 & 3 : Tokenisation comparative et préparation des entrées ---")

# Initialisation des deux architectures de tokeniseurs demandées
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-uncased")
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

sample_sentence = "L'intelligence artificielle transforme le traitement de la langue."

print(f"\nPhrase échantillon : '{sample_sentence}'")
print(f"Taille du vocabulaire BERT : {bert_tokenizer.vocab_size}")
print(f"Taille du vocabulaire XLM-R: {xlmr_tokenizer.vocab_size}")

# Exemple de traitement avec la syntaxe d'appel moderne
MAX_LEN = 16

bert_encoded = bert_tokenizer(
    sample_sentence,
    add_special_tokens=True,
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True,
    return_attention_mask=True,
    return_tensors="np"
)

xlmr_encoded = xlmr_tokenizer(
    sample_sentence,
    add_special_tokens=True,
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True,
    return_attention_mask=True,
    return_tensors="np"
)

print("\n--- Analyse des Tenseurs Générés (BERT) ---")
print(f"• Input IDs      : {bert_encoded['input_ids']}")
print(f"• Attention Mask : {bert_encoded['attention_mask']}")
print(f"• Décodage texte : {bert_tokenizer.decode(bert_encoded['input_ids'])}")

print("\n--- Analyse des Tenseurs Générés (XLM-RoBERTa) ---")
print(f"• Input IDs      : {xlmr_encoded['input_ids']}")
print(f"• Attention Mask : {xlmr_encoded['attention_mask']}")
print(f"• Décodage texte : {xlmr_tokenizer.decode(xlmr_encoded['input_ids'])}")
print(f"• Cartographie des jetons spéciaux : {xlmr_tokenizer.special_tokens_map}")

# =====================================================================
# 5. CRÉATION DES PLIS DE VALIDATION CROISÉE (STRATIFIED K-FOLD)
# =====================================================================
print("\n--- Étape 5 : Configuration de la Validation Croisée ---")

X = df['text'].values
y = df['label'].values

# n_splits ajusté à 3 pour correspondre mathématiquement au volume de notre échantillon
N_SPLITS = 3
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

train_folds = []
val_folds = []

# Extraction et stockage des ensembles d'entraînement et de validation
for fold_idx, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train_fold, X_val_fold = X[train_index], X[val_index]
    y_train_fold, y_val_fold = y[train_index], y[val_index]

    # Stockage sous forme de dictionnaires structurés dans nos listes
    train_folds.append({'text': X_train_fold, 'label': y_train_fold})
    val_folds.append({'text': X_val_fold, 'label': y_val_fold})

    print(f"• Pli n°{fold_idx+1} configuré | Taille Entraînement : {len(train_index)} | Taille Validation : {len(val_index)}")

print(f"\n✅ Félicitations ! Vos {N_SPLITS} plis de validation croisée sont prêts et équilibrés.")


--- Étape 1 & 4 : Chargement et Exploration du jeu de données ---
Forme du jeu de données (Shape) : (15, 2)

Aperçu des premières lignes :


,text,label
0,L'équipe a remporté la coupe du monde après un...,0
1,Le cours de cette action en bourse a chuté de ...,1
2,Le nouveau modèle de smartphone intègre une pu...,2
3,Le joueur a marqué un but décisif à la dernièr...,0
4,Les investisseurs craignent une hausse des tau...,1



--- Étape 2 & 3 : Tokenisation comparative et préparation des entrées ---

Phrase échantillon : 'L'intelligence artificielle transforme le traitement de la langue.'
Taille du vocabulaire BERT : 105879
Taille du vocabulaire XLM-R: 250002

--- Analyse des Tenseurs Générés (BERT) ---
• Input IDs      : [[  101   154   112 19334 33005 61463 14205 64881 10130 44724 10102 10106
  20219   119   102     0]]
• Attention Mask : [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]]
• Décodage texte : ["[CLS] l ' intelligence artificielle transforme le traitement de la langue. [SEP] [PAD]"]

--- Analyse des Tenseurs Générés (XLM-RoBERTa) ---
• Input IDs      : [[     0    339     25 130687 155269   2118  27198     13     95  66744
       8     21  79897      5      2      1]]
• Attention Mask : [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]]
• Décodage texte : ["<s> L'intelligence artificielle transforme le traitement de la langue.</s><pad>"]
• Cartographie des jetons spéciaux : {'bos_token': '<s>', 'eos_token': '</s>', 'unk_

- Rôle d'un tokeniseur de type Transformer (Étape 1 & 2) : Les réseaux de neurones ne savent pas traiter des chaînes de caractères brutes. Le tokeniseur découpe le texte en sous-mots (subwords) pour éviter le problème des mots inconnus (out-of-vocabulary), puis convertit chaque segment en un identifiant numérique unique (input_ids) référencé dans son dictionnaire de pré-entraînement [Scribd].

- Différence BERT vs XLM-RoBERTa (Étape 3) : BERT utilise des marqueurs structuraux comme [CLS] (début de séquence) et [SEP] (séparateur), alors que XLM-RoBERTa emploie le protocole SentencePiece avec des balises de phrases de type <s> et </s>. De plus, le vocabulaire de XLM-R est beaucoup plus volumineux (plus de 250 000 jetons) car il est nativement optimisé pour gérer plus de 100 langues simultanément.

- Utilité du masque d'attention (attention_mask) : Pour traiter les phrases par lots de tailles homogènes, on applique un bourrage (padding) avec des valeurs neutres (0). Le masque d'attention contient des 1 pour les vrais mots et des 0 pour les jetons de remplissage. Cela indique explicitement au mécanisme d'auto-attention du Transformer d'ignorer complètement les zones de bourrage lors du calcul des poids contextuels, ce qui évite de fausser les gradients.

- Avantage de la validation croisée stratifiée (StratifiedKFold) (Étape 5) : Contrairement à un découpage aléatoire simple qui pourrait isoler toutes les instances d'une classe rare dans un seul ensemble, la stratification garantit que chaque pli contient exactement la même proportion de classes que le jeu de données d'origine. Cela assure une évaluation impartiale et robuste des performances réelles du modèle.

